# Cryptocurrency Market Analysis — Methodology Walkthrough

This notebook reproduces the core analysis behind the dashboard, step by step:
data loading, volatility & risk, market-cycle identification, cross-asset
correlation, and ARIMA forecasting. It's meant as the *explainable* companion to
the interactive app — run top to bottom.

> Run from the project root so the `src` package is importable.

In [1]:
import sys, os
# Ensure project root is on the path when running from notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.abspath(".."))
import pandas as pd, numpy as np
import config
from src import pipeline
from src.analysis import volatility, cycles, correlation, forecast
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load market data

`load_market_data` pulls crypto (CoinGecko), macro assets (Yahoo Finance),
sentiment (Fear & Greed) and on-chain metrics (Blockchain.info), with a
transparent offline fallback. The `sources` dict records which feeds were live.

In [2]:
data = pipeline.load_market_data(days=365)
print("Assets:", data.symbols)
print("Sources:")
for feed, src in data.sources.items():
    print(f"  - {feed}: {src}")
data.crypto_frames["BTC"].tail()

Assets: ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'ADA']
Sources:
  - Crypto (CoinGecko): sample
  - Macro (Yahoo Finance): sample
  - Sentiment (Fear & Greed): sample
  - On-chain (Blockchain.info): sample


,price,volume,market_cap
2026-08-01,"29,784.2973","43,472,550,050.4506","582,428,856,659.8899"
2026-08-02,"29,739.4849","31,776,896,269.7852","581,552,554,466.4819"
2026-08-03,"30,390.8458","59,116,326,134.7215","594,289,850,129.9497"
2026-08-04,"31,996.4421","35,603,446,083.7380","625,687,119,256.0990"
2026-08-05,"31,100.8276","62,362,372,600.9857","608,173,470,417.9087"


## 2. Volatility & risk assessment

For each asset we compute annualised volatility, one-day VaR/CVaR, max drawdown,
and the Sharpe/Sortino ratios, then classify the current volatility regime.

In [3]:
rows = [volatility.risk_profile(s, data.crypto_frames[s]["price"]).as_dict()
        for s in data.symbols]
pd.DataFrame(rows).set_index("Symbol")

,Annual Vol,VaR 95%,VaR 99%,CVaR 95%,Max Drawdown,Sharpe,Sortino,Vol Regime
Symbol,,,,,,,,
BTC,62.4%,5.03%,7.02%,6.31%,-69.1%,-0.35,-0.62,Low
ETH,83.3%,6.95%,9.15%,8.27%,-61.2%,-0.54,-1.00,Low
SOL,106.2%,9.03%,10.76%,10.18%,-76.9%,-0.64,-1.19,Low
BNB,70.6%,5.71%,8.31%,7.25%,-31.0%,1.06,1.84,Elevated
XRP,96.2%,7.92%,11.66%,9.86%,-76.3%,0.95,1.67,Elevated
ADA,95.3%,8.03%,10.63%,9.85%,-69.4%,-0.13,-0.22,High


In [4]:
# Rolling annualised volatility for BTC
rv = volatility.rolling_volatility(data.crypto_frames["BTC"]["price"])
rv.tail().apply(lambda x: f"{x:.1%}")

2026-08-01    51.9%
2026-08-02    51.6%
2026-08-03    51.5%
2026-08-04    55.0%
2026-08-05    55.7%
Freq: D, Name: price, dtype: str

## 3. Market-cycle phase identification

Phases follow the Accumulation → Markup → Distribution → Markdown cycle, inferred
from price vs the 50/200-day moving averages, RSI momentum, and drawdown. The rule
set is transparent, so every label is explainable.

In [5]:
cyc = [cycles.current_phase(s, data.crypto_frames[s]["price"]).as_dict()
       for s in data.symbols]
pd.DataFrame(cyc).set_index("Symbol")

,Cycle Phase,RSI(14),Price vs 200MA,Drawdown vs ATH,Trend
Symbol,,,,,
BTC,Transition,50,+13.5%,-40.7%,Up
ETH,Accumulation,45,-14.9%,-53.0%,Down
SOL,Markdown (Bear),30,-26.0%,-74.7%,Down
BNB,Transition,53,-0.6%,-22.9%,Down
XRP,Transition,40,+42.4%,-24.1%,Up
ADA,Transition,53,-17.8%,-54.4%,Down


In [6]:
# How much time BTC spent in each phase over the window
cycles.phase_series(data.crypto_frames["BTC"]["price"]).value_counts()

phase
Undetermined       199
Accumulation        54
Transition          51
Markup (Bull)       30
Distribution        18
Markdown (Bear)     13
Name: count, dtype: int64

## 4. Digital vs traditional correlation

We align crypto and macro returns and measure co-movement. Correlations in crypto
are unstable, so we also look at the rolling view and beta.

In [7]:
rets = correlation.align_returns(data.crypto_prices, data.macro_prices)
correlation.crypto_macro_summary(rets, data.symbols, list(data.macro_prices.columns))

,S&P 500,Gold,US Dollar Index,US 10Y Yield,Crude Oil
BTC,0.0100,0.0100,-0.0200,-0.0100,0.1200
ETH,-0.0300,0.0400,0.0200,0.0800,-0.0700
SOL,0.1000,-0.0200,0.1000,-0.0300,-0.0100
BNB,-0.0100,-0.0400,0.0800,-0.0300,0.0100
XRP,-0.0200,0.0100,0.0900,0.0600,-0.0600
ADA,0.0400,-0.0500,0.0400,-0.0700,0.0300


In [8]:
# Rolling 30-day correlation: BTC vs S&P 500, plus beta
series = correlation.rolling_correlation(rets, "BTC", "S&P 500", window=30)
print("Latest 30d corr BTC vs S&P 500:", round(series.iloc[-1], 3))
print("Beta of BTC to S&P 500:", round(correlation.beta(rets, "BTC", "S&P 500"), 3))

Latest 30d corr BTC vs S&P 500: 0.169
Beta of BTC to S&P 500: 0.038


## 5. Time-series forecasting (ARIMA)

We model log-price with ARIMA: order chosen by AIC on a training split, error
measured out-of-sample, then refit on the full history to project ahead. Crypto is
close to a random walk, so the interval is honest uncertainty, not a target.

In [9]:
fr = forecast.forecast_price(data.crypto_frames["BTC"]["price"], "BTC", horizon=30)
print("Selected ARIMA order:", fr.order)
print(f"Out-of-sample  RMSE=${fr.rmse:,.0f}  MAE=${fr.mae:,.0f}  MAPE={fr.mape:.2f}%")
fr.forecast.head()

Selected ARIMA order: (2, 1, 0)
Out-of-sample  RMSE=$1,143  MAE=$912  MAPE=2.92%


2026-08-06   31,240.0632
2026-08-07   31,143.9643
2026-08-08   31,154.9749
2026-08-09   31,144.3348
2026-08-10   31,145.0378
Freq: D, Name: forecast, dtype: float64

## 6. Takeaways

- The **risk table** ranks assets by volatility and tail risk (VaR/CVaR) for sizing.
- The **cycle labels** give a phase read to contextualise entries/exits.
- The **correlation view** shows whether crypto is trading as a risk asset or a hedge.
- The **forecast** quantifies near-term uncertainty rather than promising direction.

The interactive versions of all of these live in `app.py` — run `streamlit run app.py`.